<a href="https://colab.research.google.com/github/TommySnyd/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-18%20%E2%80%94%20Pandas%20Challenge%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
# TODO - done
# created a new value of df['revenue] as price * qty for each value in the df
df['revenue'] = df['price']*df['qty']
total_revenue = df['revenue'].sum()
total_qty = df['qty'].sum()
print(f'Total revenue: ${total_revenue:.2f}')
print(f'Total quantity:  {total_qty}')

Total revenue: $8520.00
Total quantity:  783


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
# TODO - done
# used the groupby fynction and sorted them in descending order of share pct of revenue
by_category = (df.groupby('category', as_index = False)['revenue'].sum().sort_values('revenue', ascending= False))
by_category['share pct'] = (by_category['revenue']/total_revenue * 100).round(2)
by_category

,category,revenue,share pct
1,Food,4293.0,50.39
2,Merch,1771.5,20.79
0,Drink,1554.0,18.24
3,RainGear,901.5,10.58


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
# TODO - done
# used the groupby function and aggregated the revenue, qty, and price for each vendor_id
by_vendor = (df.groupby('vendor_id')
               .agg(avg_order_revenue=('revenue', 'mean'),
                    orders=('revenue', 'size'),
                    total_revenue = ('revenue', 'sum'))
               .sort_values('avg_order_revenue', ascending=False).round(2))
df

,vendor_id,category,qty,price,revenue
0,V-10,Drink,2,24.0,48.0
1,V-18,RainGear,1,12.0,12.0
2,V-18,Drink,3,4.5,13.5
3,V-10,Food,2,12.0,24.0
4,V-18,Drink,3,7.5,22.5
...,...,...,...,...,...
395,V-18,Merch,1,12.0,12.0
396,V-01,Merch,2,24.0,48.0
397,V-10,Food,3,7.5,22.5
398,V-18,Merch,2,24.0,48.0


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
# TODO - done
# used df.loc to located the "merch" and "revenue" section of each category.
share_merch = df.loc[df['category'] == 'Merch', 'revenue'].sum()/total_revenue * 100
print(f'Merch is {share_merch:.2f}% of revenue')


Merch is 20.79% of revenue


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})
joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')
print(f'Rows before/After merge: {len(df)}/ {len(joined)}')
print(f'Revenue before/after merge: {df['revenue'].sum()}/ {joined['revenue'].sum()}')
unmatched = joined.loc[joined['vendor_name'].isna(), 'vendor_id'].unique()
n_unmatched = joined['vendor_name'].isna().sum()
rev_unmatched = joined.loc[joined['vendor_name'].isna(), 'revenue'].sum()
print(f'Unmatched: {unmatched}-{n_unmatched} rows, missing revenue: {rev_unmatched:.2f}')
joined['vendor_name']=joined['vendor_name'].fillna('Missing - V18')

# TODO: merge, validate, and report the unmatched vendor - done
# used the join function to merge the vendor_names and the data frame tables based on vendor_id
# then wrote the unmatched functions to locate the values that are na in the 'vendor_name' and 'revenue' sections

Rows before/After merge: 400/ 400
Revenue before/after merge: 8520.0/ 8520.0
Unmatched: ['V-18']-108 rows, missing revenue: 2349.00


**The unmatched vendor, and what I did about it:** _..._

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
# TODO - done
# created a new value named pivot using the .pivot_table function, assigned the following values to match the assignment
pivot = joined.pivot_table(index='vendor_name', columns='category',
                     values='revenue', aggfunc='sum',
                     margins=True, margins_name='Total', fill_value=0)
pivot.round(2)

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Missing - V18,582.0,1018.5,508.5,240.0,2349.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

1 - I would tell these vendors that they all need to focus primarily on improving their drink sales because that is where all of the vendors are lacking on sales. However, they should also focus on their strengths which for all of them is food.


2 - I think my answer to my unmatched function on question 5 is the least trustworthy, mostly because it gave me the most trouble and I had to toubleshoot it the most. Also the isna function followed by the category and then a count of the unique could be an innacurate count of the true missing values if we were truly the data.